# Hotel Reservations - предсказание отмены бронирования номеров в отелях

Отели теряют деньги, когда гости отменяют бронирование в последний момент. Заказчик хочет получить модель, которая будет предсказывать потенциальные отмены, чтобы лучше управлять доходами и заполняемостью номеров.


In [1]:
# Установка зависимостей
!pip install -r requirements.txt -q

In [2]:
import pandas as pd

### 1.1 Обзор данных

In [3]:
# Загрузка датасета с кешированием на диске
try:
    df = pd.read_csv('./1. hotel.csv')
    print('Dataset found locally.')
except:
    print('Dataset not found locally.')
    df = pd.read_csv('https://code.s3.yandex.net/datasets/ds14_hotel.csv', index_col=0)
    df.to_csv('./1. hotel.csv')

Dataset found locally.


In [4]:
from sklearn.model_selection import train_test_split

# Разделяем на признаки и целевую переменную
X = df.drop(["Booking_ID", "booking_status"], axis=1)
y = df["booking_status"]

# Разделяем на train и test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
from sklearn.impute import SimpleImputer

# Признаки с пропусками
numerical_with_nan = ['no_of_weekend_nights', 'no_of_week_nights', 'required_car_parking_space']
X_with_nan = df[numerical_with_nan]

EMPTY_FEATURE_VALUE = -999999

# SimpleImputer с fill_value=-999999
imputer = SimpleImputer(fill_value=EMPTY_FEATURE_VALUE)

In [6]:
from sklearn.preprocessing import OneHotEncoder

# Обычные категориальные признаки
categorical_features = ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type']
binary_features = ['is_vip_guest']

# OneHotEncoder для обычных категориальных признаков
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# OneHotEncoder для бинарного признака с параметром drop='first'
binary_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

In [7]:
from sklearn.compose import ColumnTransformer

# ColumnTransformer с тремя трансформерами
preprocessor = ColumnTransformer([
    ('binary', binary_encoder, binary_features),
    ('categorical', categorical_encoder, categorical_features),
    ('imputer', imputer, numerical_with_nan)
], remainder='passthrough')

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Инициализируем модель
model = DecisionTreeClassifier(max_depth=10, random_state=42)

# Создадим Pipeline с preprocessor и DecisionTreeClassifier
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

# Обучим pipeline
pipeline.fit(X_train, y_train)

# Получите предсказания (классы)
y_pred = pipeline.predict(X_test)

# Получите вероятности для ROC-AUC
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Рассчитайте метрики
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy на тестовой выборке: {accuracy:.3f}")
print(f"ROC-AUC на тестовой выборке: {roc_auc:.3f}")

Accuracy на тестовой выборке: 0.847
ROC-AUC на тестовой выборке: 0.919
